# RAG

In [ ]:
%pip install langchain-ollama langchain-community langchain-huggingface langchain-core langchain-experimental faiss-cpu huggingface-hub pypdf sentence-transformers

In [9]:
# Librerías estándar
import os

# LangChain
from langchain_ollama import OllamaLLM
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from langchain_experimental.text_splitter import SemanticChunker

from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank
from langchain_core.runnables import RunnableLambda


## 1. Configuración del modelo LLM

In [10]:
llm = OllamaLLM(model="llama3.2:1b")

## 2. Carga de documentos

In [22]:
# Definimos la ruta de la carpeta
pdf_folder_path = "./data/"
docs = []

# Iteramos sobre los archivos en la carpeta 'data'
for file in os.listdir(pdf_folder_path):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(pdf_folder_path, file))
        docs.extend(loader.load())

print(f"✅ Se cargaron {len(docs)} páginas de los PDFs.")

✅ Se cargaron 3 páginas de los PDFs.


## 3. Configuración del modelo de embeddings

In [23]:
print("🔄 Descargando y cargando 'multilingual-e5-small'...")
print("Este modelo es ideal para español y eficiente en CPU/RAM.")

# Usamos el modelo E5 que es excelente para tareas de recuperación
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small"
)

print("✅ Modelo de embeddings actualizado correctamente.")

INFO:sentence_transformers.base.model:No device provided, using cpu


🔄 Descargando y cargando 'multilingual-e5-small'...
Este modelo es ideal para español y eficiente en CPU/RAM.


INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from intfloat/multilingual-e5-small.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/6142

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/multilingual-e5-small/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/intfloat/multilingual-e5-small/614241f622f53c4eeff9890bdc4f31cfecc418b3/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/intfloat/mult

✅ Modelo de embeddings actualizado correctamente.


## 4. División en fragmentos

In [24]:
# Ahora el Chunker usará la comprensión multilingüe de E5
text_splitter = SemanticChunker(
    embeddings_model, 
    breakpoint_threshold_type="percentile"
)

splits = text_splitter.split_documents(docs)

print(f"✅ Los PDFs se han vuelto a procesar con el nuevo modelo.")
print(f"✅ Total de fragmentos optimizados: {len(splits)}")

✅ Los PDFs se han vuelto a procesar con el nuevo modelo.
✅ Total de fragmentos optimizados: 7


In [25]:
# Verificamos la diferencia de tamaño entre fragmentos
if splits:
    print(f"\n📏 Tamaño del primer fragmento: {len(splits[0].page_content)} caracteres.")
    print("📄 Contenido del primer fragmento:")
    print("-" * 30)
    print(splits[0].page_content)


📏 Tamaño del primer fragmento: 1944 caracteres.
📄 Contenido del primer fragmento:
------------------------------
El  Mundo  en  Alerta:  Científicos  Confirman  que  los  Patos  Mandarines  Han  Iniciado  la  
Construcción
 
de
 
sus
 
Propias
 
Civilizaciones
  Un  estudio  de  18  meses  revela  asentamientos  jerarquizados,  sistemas  de  drenaje  y  una  
incipiente
 
escritura
 
pictográfica
 
en
 
humedales
 
de
 
China,
 
Japón
 
y
 
Reino
 
Unido. ---   Por  Ana  Karina  Mendoza,  enviada  especial  al  Humedal  de  Yancheng    *Fecha  de  publicación:  14  de  abril  de  2026  –  Actualización:  16  de  abril  de  2026*   Yancheng,  China. –  Lo  que  comenzó  como  una  anécdota  peculiar  en  foros  de  observación  
de
 
aves
 
se
 
ha
 
convertido,
 
según
 
un
 
comunicado
 
urgente
 
de
 
la
 
Unión
 
Internacional
 
para
 
la
 
Conservación
 
de
 
la
 
Naturaleza
 
(UICN),
 
en
 
uno
 
de
 
los
 
descubrimientos
 
etológicos
 
más
 
disruptivos
 
del
 
siglo. Un
 
cons

## 5. Almacén vectorial FAISS

In [26]:
# 2. Creamos el almacén vectorial (Vector Store)
# Esto toma tus 'splits' de la Fase 3 y los convierte en vectores usando el modelo
print("🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...")
vectorstore = FAISS.from_documents(splits, embeddings_model)
print("✅ Almacén vectorial creado exitosamente.")

🧠 Generando vectores e indexando en FAISS (esto puede tardar unos segundos)...
✅ Almacén vectorial creado exitosamente.


In [27]:
# ============================================
# === NUEVA CASILLA: Re‑Ranker con FlashRank
# ============================================

# 1. Base retriever: FAISS devuelve 10 fragmentos
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# 2. Compresor: FlashRank reordena y selecciona los 3 mejores
compressor = FlashrankRerank(top_n=3) # type: ignore

# 3. Retriever personalizado que hace FAISS + compresión
def retrieve_and_compress(query: str):
    """Recupera los 10 fragmentos y los comprime a los 3 más relevantes."""
    docs = base_retriever.invoke(query)
    return compressor.compress_documents(docs, query)

compression_retriever = RunnableLambda(retrieve_and_compress)

print("✅ Re‑Ranker configurado: FAISS (k=10) → FlashRank (top_n=3)")

✅ Re‑Ranker configurado: FAISS (k=10) → FlashRank (top_n=3)


## 6. Definición del pipeline RAG

In [47]:
# 1. Definimos el Prompt
template = """Eres un asistente técnico experto. Tu tarea es responder preguntas basándote EXCLUSIVAMENTE en los fragmentos de contexto proporcionados
a continuación. Sigue estas reglas estrictamente:

- Si el contexto contiene la información, responde de manera clara, concisa y directa.
- Si la pregunta no puede responderse completamente con el contexto, di: "No se encontró información suficiente en los documentos para responder esta pregunta."
- No uses conocimientos previos ni información que no aparezca explícitamente en el contexto.
- No inventes datos, fechas, nombres ni conclusiones. Si el contexto no especifica algo, indícalo.
- Organiza la respuesta en párrafos cortos, usando viñetas si es necesario para listar elementos.
- Cuando cites una fuente, menciona el número de fragmento o la página si la tienes, de lo contrario simplemente responde.

Contexto:
{context}

Pregunta: {question}

Respuesta:"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Configuramos el flujo (Chain)
# Este pipeline: toma la pregunta -> busca en FAISS -> llena el prompt -> le pregunta a Llama
rag_chain = (
    {"context": compression_retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 7. Prueba final

In [48]:
pregunta_final = "¿En qué revista científica fue publicado el estudio y en qué fecha?"
print(f"Preguntando: {pregunta_final}\n")

respuesta_final = rag_chain.invoke(pregunta_final)
print("--- RESPUESTA DEL RAG ---")
print(respuesta_final)

Preguntando: ¿En qué revista científica fue publicado el estudio y en qué fecha?



INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


--- RESPUESTA DEL RAG ---
No se encontró información suficiente en los documentos para responder esta pregunta.
